In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import splink
from splink.duckdb.linker import DuckDBLinker
from splink.datasets import splink_datasets
import pandas as pd

os.chdir(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_r_rraa_sin_snsa.parquet')

#rraa = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\rraa.parquet')
censo = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\censo.parquet')
extranjeros_vinculados = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\extranjeros_vinculados.parquet')

linker = DuckDBLinker([df_l, df_r])
linker.load_model(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\modelos\modelo4.json')

## Reglas de bloqueo

1. documento uruguayo
2. primer nombre, segundo nombre, primer apellido, segundo apellido
3. año de nacimiento, primer nombre, primer apellido
4. departamento_ajustado, año de nacimiento, primer nombre y primera letra del primer apellido
5. departamento_ajustado y fecha de nacimiento

donde en departamento_ajustado se unificaron los departamentos de Montevideo y Canelones, considerándolos como el mismo.

El siguiente gráfico presenta la cantidad de comparaciones que surgen de cada regla.

In [2]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on

settings = {"link_type": "link_only"}

linker_br = DuckDBLinker([df_r, df_l], settings)

br1_a = block_on(["documento"])
#br1_b = block_on(["documento_extranjero"])
#br2 = block_on(["documento", "documento_extranjero", "fecha_nacimiento"])
br3 = block_on(["fecha_nacimiento", "primer_nombre", "primer_apellido"])
#br4 = block_on(["primer_nombre", "primer_apellido"]) (433,314,151)
br5 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido"])
#br6 = block_on(["primer_nombre", "primer_apellido", "id_sexo"]) (430,033,729)
#br7 = block_on(["fecha_nacimiento", "id_sexo"]) (231,723,037)
br8 = block_on(["ano_nacimiento", "primer_nombre", "primer_apellido"])
br9 = block_on(["mes_nacimiento", "dia_nacimiento", "primer_nombre", "primer_apellido"])
#br10 = block_on(["id_departamento", "primer_nombre", "primer_apellido"]) (63,427,012)
#br11 = block_on(["id_departamento", "ano_nacimiento"]) (29,316,195,010)
br12 = block_on(["id_departamento", "fecha_nacimiento"]) #(83,471,400)
#br13 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)"]) (13,581,242,609)
#br14 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento"]) (292,595,639)
#br15 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)", "ano_nacimiento"]) (161,287,979)
br16 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br17 = block_on(["id_departamento_ajustado", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br18 = block_on(["id_departamento_ajustado", "fecha_nacimiento"])

blocking_rules = [br1_a, br5, br8, br17, br18]

linker_br.cumulative_num_comparisons_from_blocking_rules_chart(blocking_rules)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

## Profile columns

Se muestra una distribución de las variables, lo cual permite observar la frecuencia de los posibles valores y evaluar el uso de term frequency adjustments.

In [5]:
linker.profile_columns(['id_departamento_ajustado', 'fecha_nacimiento', 'primer_nombre'], bottom_n=3)

alt.VConcatChart(...)

## Match Weights

In [3]:
linker.match_weights_chart()

alt.VConcatChart(...)

## Term Frequency Adjustments

In [4]:
import warnings
warnings.filterwarnings("ignore")

linker.tf_adjustment_chart("id_departamento_ajustado")

alt.HConcatChart(...)